In [ ]:
import json
from pathlib import Path
from collections import Counter

BASE = Path("..")

INPUT_USERNAMES = BASE / "complete_dataset_raw/raw_usernames_compressed.jsonl"
INPUT_NUMBERS = BASE / "complete_dataset_raw/raw_numbers_compressed.jsonl"

OUTPUT_DIR = BASE / "dataset_filtered"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_USERNAMES = OUTPUT_DIR / "usernames_filtered.jsonl"
OUTPUT_NUMBERS = OUTPUT_DIR / "numbers_filtered.jsonl"

KEEP_TYPES = {"nft_transfer", "nft_purchase"}

In [2]:
def filtra_actions(input_path, output_path):
    stats = Counter()
    kept_types = Counter()

    with open(input_path, "r", encoding="utf-8") as fin, \
         open(output_path, "w", encoding="utf-8") as fout:

        for line in fin:
            if not line.strip():
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                stats["json_non_validi"] += 1
                continue

            stats["totale"] += 1

            action_type = record.get("y")
            success = record.get("ok")

            if action_type not in KEEP_TYPES:
                stats["esclusi_tipo"] += 1
                continue

            if success is not True:
                stats["esclusi_success"] += 1
                continue

            # Scrive la riga originale senza modifiche
            fout.write(line)

            stats["mantenuti"] += 1
            kept_types[action_type] += 1

    print(f"\nFILE: {input_path.name}")
    print("-" * 45)
    print(f"Actions totali: {stats['totale']:,}")
    print(f"Actions mantenute: {stats['mantenuti']:,}")
    print(f"Escluse per action_type: {stats['esclusi_tipo']:,}")
    print(f"Escluse per success: {stats['esclusi_success']:,}")
    print(f"JSON non validi: {stats['json_non_validi']:,}")

    print("\nAction type mantenuti:")
    for tipo, count in kept_types.items():
        print(f"  {tipo}: {count:,}")

    print(
        "\nDimensione output:",
        round(output_path.stat().st_size / 1024**2, 2),
        "MB"
    )

    return stats, kept_types

In [3]:
# USERNAMES
stats_usernames, kept_usernames = filtra_actions(
    INPUT_USERNAMES,
    OUTPUT_USERNAMES
)

# PHONE NUMBERS
stats_numbers, kept_numbers = filtra_actions(
    INPUT_NUMBERS,
    OUTPUT_NUMBERS
)


FILE: raw_usernames_compressed.jsonl
---------------------------------------------
Actions totali: 41,442,208
Actions mantenute: 2,579,153
Escluse per action_type: 38,824,857
Escluse per success: 38,198
JSON non validi: 0

Action type mantenuti:
  nft_transfer: 2,500,782
  nft_purchase: 78,371

Dimensione output: 3542.43 MB



FILE: raw_numbers_compressed.jsonl
---------------------------------------------
Actions totali: 12,089,699
Actions mantenute: 837,146
Escluse per action_type: 11,228,032
Escluse per success: 24,521
JSON non validi: 0

Action type mantenuti:
  nft_transfer: 789,835
  nft_purchase: 47,311

Dimensione output: 1196.28 MB


In [1]:
import json
import pandas as pd
from pathlib import Path
from collections import Counter

FILES = {
    "usernames": Path("dataset_filtered/usernames_filtered.jsonl"),
    "phone_numbers": Path("dataset_filtered/numbers_filtered.jsonl")
}

risultati = []

for layer, path in FILES.items():
    nft_unici = set()
    tipi = Counter()
    totale_righe = 0

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            record = json.loads(line)
            totale_righe += 1

            # NFT address (campo compresso)
            nft = record.get("n")

            if nft:
                nft_unici.add(nft.lower())

            tipi[record.get("y")] += 1

    risultati.append({
        "layer": layer,
        "totale_righe": totale_righe,
        "nft_distinti": len(nft_unici),
        "nft_transfer": tipi["nft_transfer"],
        "nft_purchase": tipi["nft_purchase"]
    })

df_riepilogo = pd.DataFrame(risultati)

display(df_riepilogo)

ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/home/local/ADUNIPI/a.belliani/.local/lib/python3.10/site-packages/traitlets/traitlets.py", line 651, in get
    value = obj._trait_values[self.name]
KeyError: '_control_lock'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/local/ADUNIPI/a.belliani/.local/lib/python3.10/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "/home/local/ADUNIPI/a.belliani/.local/lib/python3.10/site-packages/ipykernel/kernelbase.py", line 347, in dispatch_control
    async with self._control_lock:
  File "/home/local/ADUNIPI/a.belliani/.local/lib/python3.10/site-packages/traitlets/traitlets.py", line 706, in __get__
    return self.get(obj, cls)  # type:ignore[return-value]
  File "/home/local/ADUNIPI/a.belliani/.local/lib/python3.10/site-packages/traitlets/traitlets.py", line 668, in get
    val

,layer,totale_righe,nft_distinti,nft_transfer,nft_purchase
0,usernames,2579153,555207,2500782,78371
1,phone_numbers,837146,136566,789835,47311
